# Biology — Microscopy cell counting and morphology
## FDP Day 5 — AI-Powered Scientific Research with Google Colab

**Objective:** Segment a microscopy image, estimate visible object count and explore morphology measurements. Threshold-based counting needs expert validation.

**Matching dataset:** `02_Biology_Microscopy_Cell_Morphology.csv`

All bundled CSVs are **synthetic teaching datasets** with plausible ranges and units, not real experimental observations. Replace them with approved research data for actual studies. Avoid uploading identifiable patient or confidential data.

**Workflow:** Upload → inspect → analyse → visualise → interpret → export.


## 1. Setup and upload the matching CSV

In [ ]:
!pip -q install seaborn scikit-learn opencv-python-headless
import io, warnings, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from google.colab import files
from IPython.display import display
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid",context="notebook")
plt.rcParams["figure.figsize"]=(9,5)
uploaded=files.upload()
csv_name=next(f for f in uploaded if f.lower().endswith(".csv"))
df=pd.read_csv(io.BytesIO(uploaded[csv_name]))
print("Loaded",csv_name,"shape:",df.shape)
display(df.head()); display(df.describe(include="all").T)


## 2. Upload image input
**Suggested image:** `02_sample_microscopy_cells.png`. Supported formats: JPG, JPEG, PNG. Use original-resolution, focused images with consistent lighting/magnification. The supplied image is an illustrative teaching sample.

In [ ]:
from PIL import Image
image_upload=files.upload()
image_name=next(f for f in image_upload if f.lower().endswith((".jpg",".jpeg",".png")))
img=np.array(Image.open(io.BytesIO(image_upload[image_name])).convert("RGB"))
print("Image shape:",img.shape)
plt.figure(figsize=(10,6));plt.imshow(img);plt.axis("off");plt.title("Uploaded image");plt.show()


## 3. Microscopy image segmentation and morphology

In [ ]:
import cv2
gray=cv2.cvtColor(img,cv2.COLOR_RGB2GRAY);blur=cv2.GaussianBlur(gray,(5,5),0)
binary=cv2.adaptiveThreshold(blur,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,31,7)
binary=cv2.morphologyEx(binary,cv2.MORPH_OPEN,np.ones((3,3),np.uint8))
contours,_=cv2.findContours(binary,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
items=[];out=img.copy()
for c in contours:
    ar=cv2.contourArea(c);p=cv2.arcLength(c,True)
    if 35<=ar<=1800:
        items.append([ar,p,4*np.pi*ar/(p*p) if p else np.nan]);cv2.drawContours(out,[c],-1,(0,255,0),2)
det=pd.DataFrame(items,columns=["area_pixels","perimeter_pixels","circularity"])
fig,ax=plt.subplots(1,3,figsize=(15,5))
for a,z,t in zip(ax,[img,binary,out],["Original","Mask","Detected objects"]):a.imshow(z,cmap="gray" if t=="Mask" else None);a.set_title(t);a.axis("off")
plt.tight_layout();plt.show()
print("Estimated visible objects:",len(det));display(det.describe().round(2))
if len(det): sns.histplot(det.area_pixels,bins=15);plt.title("Detected object area");plt.show()
sns.boxplot(data=df,x="sample_group",y="cell_area_um2");plt.title("Cell area by sample group");plt.show()


## Research interpretation checklist
- State the research question, data source, sample size and units.
- Explain two findings supported by the charts.
- Identify uncertainty, confounders, measurement limitations and possible bias.
- State what additional data or validation is required before publication.
- Do not claim causation from correlation or clinical/biological diagnosis from simple image segmentation.


In [ ]:
# Export a copy of the data with any new analysis columns
df.to_csv("processed_research_data.csv",index=False)
files.download("processed_research_data.csv")
